# Named Entity Recognition System

Sequence labeling pipeline that detects **PER, ORG, LOC and MISC** entities on CoNLL-2003.

**Models compared:** LSTM · BiLSTM · BiLSTM + CRF (built from scratch in PyTorch) · DistilBERT (fine-tuned with HuggingFace)

**Pipeline:** data loading → EDA → IOB tagging → vocabularies and GloVe → char-CNN for OOV words → training → seqeval evaluation → visualization → Gradio app

## 1. Setup
Install the libraries and check that the GPU is available.

In [ ]:
! pip install datasets seqeval pytorch-crf transformers accelerate gradio

import torch 
print(torch.__version__)
print("GPU", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
import gensim.downloader as api
from torch.utils.data import Dataset, DataLoader
from IPython.display import display, HTML
import torch.nn as nn
from torchcrf import CRF
from seqeval.metrics import classification_report, f1_score
import copy, time
from transformers import pipeline

## 2. Load the Dataset
CoNLL-2003 (`eriktks/conll2003`, parquet version): 14,041 train / 3,250 validation / 3,453 test sentences.

Tags use the **IOB scheme**:
- `B-X`: first token of an entity of type X
- `I-X`: continuation of the same entity
- `O`: outside any entity

CoNLL has 4 entity types: `PER`, `ORG`, `LOC`, `MISC`.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("eriktks/conll2003", revision = "refs/convert/parquet")
print(dataset)

label_list = dataset['train'].features["ner_tags"].feature.names
print(label_list)
print(dataset["train"][0])

## 3. Exploratory Data Analysis
- The data is highly imbalanced: `O` has about 169k tokens, while all entity tags together have about 34k. Accuracy is misleading here, so we evaluate with **seqeval F1**.
- The average sentence has 14.5 tokens and 95% are shorter than 37 tokens.
- We deliberately skip lowercasing, stopword removal and stemming. Capitalization is a strong signal for names, and removing tokens would break the token-to-label alignment.

In [ ]:
# 1) table: token + tag fro first sentence
ex = dataset["train"][0]
display(pd.DataFrame({
    "token": ex["tokens"],
    "tag_id": ex["ner_tags"],
    "tag": [label_list[i] for i in ex["ner_tags"]],
}).T)

# 2) 8 sentences 
COLORS = {"PER": "#ffd166", "ORG": "#8ecae6", "LOC": "#95d5b2", "MISC": "#f4a6c1"}

def show_sample(tokens, tags):
    html = ""
    for tok, t in zip(tokens, tags):
        lab = label_list[t]
        if lab == "O":
            html += f"{tok} "
        else:
            ent = lab.split("-")[1]
            html += (f"<span style='background:{COLORS[ent]};color:#000;padding:2px 5px;"
                     f"border-radius:4px;margin:1px'>{tok} <small><b>{lab}</b></small></span> ")
    return f"<p style='font-size:15px;line-height:2'>{html}</p>"

html = "".join(show_sample(dataset["train"][i]["tokens"], dataset["train"][i]["ner_tags"])
               for i in range(1, 9))
display(HTML(html))

In [ ]:
# example by the label as a text
ex = dataset['train'][0]
print(list(zip(ex['tokens'], [label_list[i] for i in ex['ner_tags']])))

# distributes tags in the train
tag_counts = Counter(label_list[t] for s in dataset["train"]["ner_tags"] for t in s )
df_tags = pd.DataFrame(tag_counts.items(), columns=["tag", "count"]).sort_values("count", ascending=False)
print(df_tags)

# distributes the longest sentences 
lengths = [len(t) for t in dataset["train"]["tokens"]]
print("max:", max(lengths), "| mean:", round(sum(lengths)/len(lengths), 1), "| 95%:", sorted(lengths)[int(0.95*len(lengths))])

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df_tags.plot.bar(x="tag", y="count", ax=ax[0], legend=False, title="Tag distribution")
ax[1].hist(lengths, bins=40); ax[1].set_title("Sentence length")
plt.tight_layout(); plt.show()

## 4. Vocabularies and GloVe Embeddings
- **Word vocab:** lowercased, to match GloVe.
- **Char vocab:** case-preserved, used by the char-CNN to handle OOV words.
- **Tag vocab:** `<PAD>` at index 0 for masking.
- **GloVe (100d)** initializes the word embeddings. Coverage is **87.6%**, and the remaining words are handled by character-level features.


In [ ]:
PAD, UNK = "<PAD>", "<UNK>"

# Word vocab (lowercase for GloVe)
word_counts = Counter(w.lower() for s in dataset["train"]["tokens"] for w in s)
word2idx = {PAD: 0, UNK: 1}
for w in word_counts:
    word2idx[w] = len(word2idx)

# Char vocab 
chars = sorted({c for s in dataset["train"]["tokens"] for w in s for c in w})
char2idx = {PAD: 0, UNK: 1}
for c in chars:
    char2idx[c] = len(char2idx)

# Tag vocab (PAD = 0 for CRF & masking)
tag2idx = {"<PAD>": 0}
for t in label_list:
    tag2idx[t] = len(tag2idx)
idx2tag = {i: t for t, i in tag2idx.items()}

print("words:", len(word2idx), "| chars:", len(char2idx), "| tags:", len(tag2idx))

# GloVe 100d
glove = api.load("glove-wiki-gigaword-100")
EMB_DIM = 100
emb_matrix = np.random.normal(0, 0.1, (len(word2idx), EMB_DIM)).astype("float32")
emb_matrix[0] = 0
found = 0
for w, i in word2idx.items():
    if w in glove:
        emb_matrix[i] = glove[w]; found += 1
print(f"GloVe coverage: {found}/{len(word2idx)} = {found/len(word2idx):.1%}")


## 5. Dataset and DataLoaders
Each sentence is converted to word ids, char ids (max 20 per word) and tag ids. Batches are padded dynamically, and a **mask** marks the real tokens so the padding is ignored in the loss and the CRF.

In [ ]:
MAX_CHARS = 20  # Max char for one word

class NERDataset(Dataset):
    def __init__(self, split):
        self.tokens, self.tags = split["tokens"], split["ner_tags"]
    def __len__(self):
        return len(self.tokens)
    def __getitem__(self, i):
        toks = self.tokens[i]
        words = [word2idx.get(w.lower(), 1) for w in toks]
        chars = [[char2idx.get(c, 1) for c in w[:MAX_CHARS]] for w in toks]
        tags  = [tag2idx[label_list[t]] for t in self.tags[i]]
        return words, chars, tags

def collate(batch):
    B, L = len(batch), max(len(b[0]) for b in batch)
    words = torch.zeros(B, L, dtype=torch.long)
    chars = torch.zeros(B, L, MAX_CHARS, dtype=torch.long)
    tags  = torch.zeros(B, L, dtype=torch.long)
    mask  = torch.zeros(B, L, dtype=torch.bool)
    for i, (w, c, t) in enumerate(batch):
        n = len(w)
        words[i, :n] = torch.tensor(w)
        tags[i, :n]  = torch.tensor(t)
        mask[i, :n]  = True
        for j, ch in enumerate(c):
            chars[i, j, :len(ch)] = torch.tensor(ch)
    return words, chars, tags, mask

train_dl = DataLoader(NERDataset(dataset["train"]),      batch_size=32, shuffle=True,  collate_fn=collate)
val_dl   = DataLoader(NERDataset(dataset["validation"]), batch_size=64, shuffle=False, collate_fn=collate)
test_dl  = DataLoader(NERDataset(dataset["test"]),       batch_size=64, shuffle=False, collate_fn=collate)

w, c, t, m = next(iter(train_dl))
print("words:", w.shape, "| chars:", c.shape, "| tags:", t.shape, "| mask:", m.shape)

## 6. Model Architecture
One configurable class builds all three from-scratch models.

**Input:** GloVe word embedding (100d) concatenated with a char-CNN feature (50d)

| Model | Bidirectional | CRF |
|---|---|---|
| LSTM | No | No |
| BiLSTM | Yes | No |
| BiLSTM + CRF | Yes | Yes |

The CRF layer learns tag-transition scores and decodes the best tag sequence with Viterbi, instead of picking each tag independently.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class NERModel(nn.Module):
    def __init__(self, emb_matrix, n_chars, n_tags, bidirectional=False, use_crf=False,
                 char_dim=30, char_filters=50, hidden=128, dropout=0.5):
        super().__init__()
        self.use_crf = use_crf
        # Word embeddings (GloVe)
        self.word_emb = nn.Embedding.from_pretrained(torch.tensor(emb_matrix), freeze=False, padding_idx=0)
        # Char CNN (to solve problem OOV)
        self.char_emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.char_cnn = nn.Conv1d(char_dim, char_filters, kernel_size=3, padding=1)
        # LSTM
        self.lstm = nn.LSTM(emb_matrix.shape[1] + char_filters, hidden,
                            batch_first=True, bidirectional=bidirectional)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * (2 if bidirectional else 1), n_tags)
        if use_crf:
            self.crf = CRF(n_tags, batch_first=True)

    def _emissions(self, words, chars):
        B, L, C = chars.shape
        c = self.char_emb(chars.view(B * L, C)).transpose(1, 2)   # (B*L, char_dim, C)
        c = torch.relu(self.char_cnn(c)).max(dim=2).values        # (B*L, filters)
        c = c.view(B, L, -1)
        x = self.drop(torch.cat([self.word_emb(words), c], dim=-1))
        out, _ = self.lstm(x)
        return self.fc(self.drop(out))

    def loss(self, words, chars, tags, mask):
        em = self._emissions(words, chars)
        if self.use_crf:
            return -self.crf(em, tags, mask=mask, reduction="mean")
        return nn.functional.cross_entropy(em.view(-1, em.size(-1)), tags.view(-1), ignore_index=0)

    @torch.no_grad()
    def predict(self, words, chars, mask):
        em = self._emissions(words, chars)
        if self.use_crf:
            return self.crf.decode(em, mask=mask)
        preds = em.argmax(-1)
        return [p[m].tolist() for p, m in zip(preds, mask)]

def build(name):
    configs = {
        "LSTM":       dict(bidirectional=False, use_crf=False),
        "BiLSTM":     dict(bidirectional=True,  use_crf=False),
        "BiLSTM_CRF": dict(bidirectional=True,  use_crf=True),
    }
    return NERModel(emb_matrix, len(char2idx), len(tag2idx), **configs[name]).to(device)

# fast test: forward + loss for each model
w, c, t, m = [x.to(device) for x in next(iter(train_dl))]
for name in ["LSTM", "BiLSTM", "BiLSTM_CRF"]:
    model = build(name)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name:11s} | params: {n_params:,} | loss: {model.loss(w, c, t, m).item():.3f}")

## 7. Training and Evaluation
Training uses Adam (lr = 1e-3), gradient clipping and 10 epochs. The checkpoint with the best **validation F1** is kept. Evaluation uses **seqeval** (entity-level precision, recall and F1).

In [ ]:
def evaluate(model, dl, report=False):
    model.eval()
    y_true, y_pred = [], []
    for w, c, t, m in dl:
        w, c, t, m = w.to(device), c.to(device), t.to(device), m.to(device)
        preds = model.predict(w, c, m)
        for p, gold, mk in zip(preds, t, m):
            gold = gold[mk].tolist()
            y_true.append([idx2tag[i] for i in gold])
            y_pred.append([idx2tag[i] for i in p])
    if report:
        print(classification_report(y_true, y_pred, digits=4))
    return f1_score(y_true, y_pred)

def train_model(name, epochs=10, lr=1e-3):
    model = build(name)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_f1, best_state, history = 0, None, []
    for ep in range(1, epochs + 1):
        model.train()
        total, t0 = 0, time.time()
        for w, c, t, m in train_dl:
            w, c, t, m = w.to(device), c.to(device), t.to(device), m.to(device)
            opt.zero_grad()
            loss = model.loss(w, c, t, m)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total += loss.item()
        val_f1 = evaluate(model, val_dl)
        history.append((total / len(train_dl), val_f1))
        if val_f1 > best_f1:
            best_f1, best_state = val_f1, copy.deepcopy(model.state_dict())
        print(f"{name} | epoch {ep:2d} | loss {total/len(train_dl):.3f} | val F1 {val_f1:.4f} | {time.time()-t0:.0f}s")
    model.load_state_dict(best_state)
    return model, history

### LSTM Model Training

In [ ]:
lstm_model, lstm_hist = train_model("LSTM", epochs=10)
torch.save(lstm_model.state_dict(), "lstm.pt")

### LSTM EVA

In [ ]:
print("=== LSTM | Test ===")
lstm_test_f1 = evaluate(lstm_model, test_dl, report=True)
print("Overall test F1:", round(lstm_test_f1, 4))

### BiLSTM Model Training

In [ ]:
bilstm_model, bilstm_hist = train_model("BiLSTM", epochs=10)
torch.save(bilstm_model.state_dict(), "bilstm.pt")

### BiLSTM EVA 

In [ ]:
print("=== BiLSTM | Test ===")
bilstm_test_f1 = evaluate(bilstm_model, test_dl, report=True)
print("Overall test F1:", round(bilstm_test_f1, 4))

### BiLSTM + CRF TRAINING

In [ ]:
crf_model, crf_hist = train_model("BiLSTM_CRF", epochs=10)
torch.save(crf_model.state_dict(), "bilstm_crf.pt")

### BiLSTM + CRF EVA

In [ ]:
print("=== BiLSTM+CRF | Test ===")
crf_test_f1 = evaluate(crf_model, test_dl, report=True)
print("Overall test F1:", round(crf_test_f1, 4))

## 8. Fine-tuning DistilBERT
`distilbert-base-cased` is fine-tuned with `AutoTokenizer`, `AutoModelForTokenClassification` and `Trainer`.

**Label alignment:** a word may be split into several subwords. Only the first subword keeps the word's label, and the others (plus `[CLS]` and `[SEP]`) get `-100`, which the loss ignores.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(batch):
    enc = tokenizer(batch["tokens"], truncation=True, is_split_into_words=True)
    all_labels = []
    for i, labels in enumerate(batch["ner_tags"]):
        prev, lab = None, []
        for wid in enc.word_ids(batch_index=i):
            if wid is None:        lab.append(-100)          # [CLS] / [SEP]
            elif wid != prev:      lab.append(labels[wid])   # first subword
            else:                  lab.append(-100)          
            prev = wid
        all_labels.append(lab)
    enc["labels"] = all_labels
    return enc

tokenized = dataset.map(tokenize_and_align, batched=True,
                        remove_columns=dataset["train"].column_names)

ex = dataset["train"][0]
enc = tokenizer(ex["tokens"], is_split_into_words=True)
toks = tokenizer.convert_ids_to_tokens(enc["input_ids"])
labs = tokenized["train"][0]["labels"]
for t, l in zip(toks, labs):
    print(f"{t:12s} {l if l == -100 else label_list[l]}")

In [ ]:
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)

id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

hf_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)

def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)
    y_true, y_pred = [], []
    for pr, lb in zip(preds, labels):
        y_true.append([label_list[l] for l in lb if l != -100])
        y_pred.append([label_list[p_] for p_, l in zip(pr, lb) if l != -100])
    return {"f1": f1_score(y_true, y_pred)}

args = TrainingArguments(
    output_dir="distilbert-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=hf_model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("distilbert-ner-final")
tokenizer.save_pretrained("distilbert-ner-final")

### EVALUATION

In [ ]:
pred_out = trainer.predict(tokenized["test"])
preds = np.argmax(pred_out.predictions, axis=-1)
labels = pred_out.label_ids

y_true, y_pred = [], []
for pr, lb in zip(preds, labels):
    y_true.append([label_list[l] for l in lb if l != -100])
    y_pred.append([label_list[p] for p, l in zip(pr, lb) if l != -100])

print("=== DistilBERT | Test ===")
print(classification_report(y_true, y_pred, digits=4))
bert_test_f1 = f1_score(y_true, y_pred)
print("Overall test F1:", round(bert_test_f1, 4))

## 9. Results and Comparison

| Model | Test F1 |
|---|---|
| LSTM | 0.7595 |
| BiLSTM | 0.8343 |
| BiLSTM + CRF | 0.8617 |
| DistilBERT | **0.8910** |

- **LSTM → BiLSTM (+7.5):** reading in both directions gives each token its right-hand context, and ORG improved the most.
- **BiLSTM → CRF (+2.7):** the CRF mainly improves precision by removing impossible tag sequences.
- **DistilBERT** is best overall thanks to pre-training, especially on PER and ORG. MISC is the hardest type for every model.

## 10. Where the CRF Improves Boundary Detection
IOB has hard rules, for example `I-PER` must follow `B-PER` or `I-PER`. A BiLSTM predicts each tag independently, so it can break these rules.

**Invalid IOB transitions on the test set:**
- BiLSTM: **180**
- BiLSTM + CRF: **7**

Example (sentence #24, "Italy recalled Marcello Cuttitta"): the BiLSTM predicted `B-ORG I-PER`, an entity that starts as ORG and continues as PER. The CRF corrected it to `B-PER I-PER`.

In [ ]:
# 1) Table and draw of comparison
results = pd.DataFrame({
    "Model": ["LSTM", "BiLSTM", "BiLSTM+CRF", "DistilBERT"],
    "Test F1": [lstm_test_f1, bilstm_test_f1, crf_test_f1, bert_test_f1],
}).round(4)
display(results)
results.plot.bar(x="Model", y="Test F1", legend=False, ylim=(0.7, 0.95), title="Test F1 comparison")
plt.xticks(rotation=0); plt.show()

# 2) predictions BiLSTM & BiLSTM+CRF on a test
def get_preds(model, dl):
    model.eval(); out = []
    for w, c, t, m in dl:
        p = model.predict(w.to(device), c.to(device), m.to(device))
        out += [[idx2tag[i] for i in x] for x in p]
    return out

gold   = [[label_list[t] for t in s] for s in dataset["test"]["ner_tags"]]
p_bi   = get_preds(bilstm_model, test_dl)
p_crf  = get_preds(crf_model, test_dl)

def invalid_iob(tags):
    bad = 0
    for i, t in enumerate(tags):
        if t.startswith("I-"):
            prev = tags[i-1] if i > 0 else "O"
            if prev == "O" or prev[2:] != t[2:]:
                bad += 1
    return bad

print("Invalid IOB transitions | BiLSTM:", sum(invalid_iob(t) for t in p_bi),
      "| BiLSTM+CRF:", sum(invalid_iob(t) for t in p_crf))

# 4) sentences of BiLSTM False and CRF True
shown = 0
for i, (g, a, b) in enumerate(zip(gold, p_bi, p_crf)):
    if a != g and b == g:
        toks = dataset["test"][i]["tokens"]
        print(f"\n#{i}")
        print("Token     :", " ".join(toks))
        print("Gold      :", " ".join(g))
        print("BiLSTM    :", " ".join(a))
        print("BiLSTM+CRF:", " ".join(b))
        shown += 1
        if shown == 4: break

## 11. Entity Visualization
Predicted entity spans are highlighted with color-coded labels: **PER** (yellow), **ORG** (blue), **LOC** (green), **MISC** (pink).

In [ ]:
ner_pipe = pipeline("token-classification", model="distilbert-ner-final",
                    aggregation_strategy="simple",
                    device=0 if torch.cuda.is_available() else -1)

COLORS = {"PER": "#ffd166", "ORG": "#8ecae6", "LOC": "#95d5b2", "MISC": "#f4a6c1"}

def highlight(text):
    ents = ner_pipe(text)
    html, last = "", 0
    for e in ents:
        html += text[last:e["start"]]
        g = e["entity_group"]
        html += (f"<span style='background:{COLORS.get(g, '#ddd')};color:#000;padding:2px 6px;"
                 f"border-radius:4px;margin:0 1px'>{text[e['start']:e['end']]} "
                 f"<small><b>{g}</b></small></span>")
        last = e["end"]
    html += text[last:]
    return f"<p style='font-size:16px;line-height:2'>{html}</p>"

samples = [
    "Barack Obama visited Microsoft headquarters in Seattle last Monday.",
    "Mohamed Salah scored for Liverpool against Manchester United in England.",
    "The European Union criticized the German government over the Egyptian deal.",
]
display(HTML("".join(highlight(s) for s in samples)))

## 12. Deployment with Gradio
The final DistilBERT model is served through a Gradio web app. Users enter free text and the app highlights the detected entities with color-coded labels, plus a table with the confidence of each entity.

Run locally with `python app.py` (needs `distilbert-ner-final/` next to `app.py`).

In [ ]:
app_code = '''
import gradio as gr
from transformers import pipeline

ner_pipe = pipeline("token-classification", model="distilbert-ner-final",
                    aggregation_strategy="simple")

COLORS = {"PER": "#ffd166", "ORG": "#8ecae6", "LOC": "#95d5b2", "MISC": "#f4a6c1"}

def predict(text):
    ents = ner_pipe(text)
    return {
        "text": text,
        "entities": [
            {"entity": e["entity_group"], "start": e["start"], "end": e["end"]}
            for e in ents
        ],
    }

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=4, label="Enter text", placeholder="Type or paste text here..."),
    outputs=gr.HighlightedText(color_map=COLORS, label="Detected entities"),
    title="Named Entity Recognition",
    description="Detects PER, ORG, LOC and MISC entities (DistilBERT fine-tuned on CoNLL-2003).",
    examples=[
        ["Barack Obama visited Microsoft headquarters in Seattle last Monday."],
        ["Mohamed Salah scored for Liverpool against Manchester United in England."],
    ],
    flagging_mode="never",
)

if __name__ == "__main__":
    demo.launch()
'''
with open("app.py", "w") as f:
    f.write(app_code)

print("app.py saved")

In [ ]:
import importlib, app
importlib.reload(app)
app.demo.launch(share=True)

## Conclusion
- Every architectural step (bidirectionality, CRF, pre-training) improved test F1.
- The CRF's main contribution is structural: it cuts invalid IOB transitions from 180 to 7.
- The fine-tuned DistilBERT is the best model (F1 = 0.891) and is the one deployed in the app.
- Known limits: MISC is the weakest class, and the model can confuse names that are both a person and a place (for example, it tagged the basketball player "Jordan" as LOC instead of PER).